In [15]:
import pandas as pd
from selenium.webdriver.common.by import By
from selenium import webdriver
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from utils.constants import CHROMEDRIVER_PATH, UU_REGULATIONS_CSV, UU_REGULATION_DETAILS_CSV, UU_DATA_DIR, UU_METADATA
from utils.scraping import download_file, generate_id, create_metadata

In [20]:
service = Service(executable_path=temp_path)
driver = webdriver.Chrome(service=service, options=chrome_options)

SessionNotCreatedException: Message: session not created: Missing or invalid capabilities
  (Driver info: chromedriver=72.0.3626.69 (3c16f8a135abc0d4da2dff33804db79b849a7c38),platform=Linux 5.15.0-119-generic x86_64)


# Scrape List of Regulations

In [1]:
regulations_xpath = '//div[@class="col-md-12"]'
title_xpath = './/p[@style="padding-top: -2;"]'
link_xpath = './/a[@title="lihat detail"]'

In [2]:
MAX_PAGES = 3
WEBSITE = "https://peraturan.go.id/uu?page="

all_regulations = []

for page in range(1, MAX_PAGES+1):
  web_page = WEBSITE + str(page)
  driver.get(web_page)
  print("web_page", web_page)

  WebDriverWait(driver, 10).until(EC.presence_of_all_elements_located((By.XPATH, regulations_xpath)))
  
  regulations = driver.find_elements(By.XPATH, regulations_xpath)

  for regulation in regulations:
    title = regulation.find_element(By.XPATH, title_xpath)
    link = regulation.find_element(By.XPATH, link_xpath)

    all_regulations.append({"title": title.text, "link": link.get_attribute('href')})

web_page https://peraturan.go.id/uu?page=1
web_page https://peraturan.go.id/uu?page=2
web_page https://peraturan.go.id/uu?page=3


In [1]:
df = pd.DataFrame(all_regulations)
df.to_csv(UU_REGULATIONS_CSV, index=False)

# Scrape Regulation Detail

In [3]:
title_div_xpath = '//div[@class="detail_title_1"]'
table_xpath = '//table[@class="table caption-top"]'
row_xpath = './/tr'
row_q_xpath = './/th[@style="width: 30%;"]'
row_a_xpath = './/td'
docs_xpath = './/a'

SKIP_COLS = ["pejabat_yang_menetapkan", "tahun_pengundangan", "nomor_pengundangan",
              "nomor_tambahan", "tanggal_pengundangan", "pejabat_pengundangan"]

In [4]:
df_all_reg = pd.read_csv(UU_REGULATIONS_CSV)

In [6]:
from utils.scraping import slugify

all_regulation_details = []

for index, row in df_all_reg.iterrows():
  print(index, row['link'])
  reg_detail = {}
  driver.get(row['link'])
  WebDriverWait(driver, 10).until(EC.presence_of_all_elements_located((By.XPATH, table_xpath)))

  title = driver.find_element(By.XPATH, title_div_xpath)
  reg_detail["title"] = title.text

  tables = driver.find_elements(By.XPATH, table_xpath)

  for table in tables:
    rows = table.find_elements(By.XPATH, row_xpath)
    for row in rows:
      row_q = row.find_element(By.XPATH, row_q_xpath)
      row_q_text = slugify(row_q.text)

      if row_q_text == "dokumen_peraturan":
        row_as = row.find_elements(By.XPATH, docs_xpath)
        row_a_texts = [row_a.get_attribute('href') for row_a in row_as]
      elif row_q_text in SKIP_COLS:
        continue
      else:
        row_a = row.find_element(By.XPATH, row_a_xpath)
        row_a_texts = row_a.text
        
      reg_detail[row_q_text] = row_a_texts
  
  all_regulation_details.append(reg_detail)

0 https://peraturan.go.id/id/uu-no-2-tahun-2024
1 https://peraturan.go.id/id/uu-no-3-tahun-2024
2 https://peraturan.go.id/id/uu-no-1-tahun-2024
3 https://peraturan.go.id/id/uu-no-21-tahun-2023
4 https://peraturan.go.id/id/uu-no-22-tahun-2023
5 https://peraturan.go.id/id/uu-no-20-tahun-2023
6 https://peraturan.go.id/id/uu-no-17-tahun-2023
7 https://peraturan.go.id/id/uu-no-19-tahun-2023
8 https://peraturan.go.id/id/uu-no-18-tahun-2023
9 https://peraturan.go.id/id/uu-no-1-tahun-2023
10 https://peraturan.go.id/id/uu-no-2-tahun-2023
11 https://peraturan.go.id/id/uu-no-3-tahun-2023
12 https://peraturan.go.id/id/uu-no-4-tahun-2023
13 https://peraturan.go.id/id/uu-no-5-tahun-2023
14 https://peraturan.go.id/id/uu-no-6-tahun-2023
15 https://peraturan.go.id/id/uu-no-7-tahun-2023
16 https://peraturan.go.id/id/uu-no-8-tahun-2023
17 https://peraturan.go.id/id/uu-no-9-tahun-2023
18 https://peraturan.go.id/id/uu-no-10-tahun-2023
19 https://peraturan.go.id/id/uu-no-11-tahun-2023
20 https://peraturan.g

In [30]:
df = pd.DataFrame(all_regulation_details)
df.to_csv(UU_REGULATION_DETAILS_CSV, index=False)

In [ ]:
## Note:
# Hubungan Antar Peraturan & Dasar Hukum not scraped yet

# Download Files

In [3]:
import ast

df = pd.read_csv(UU_REGULATION_DETAILS_CSV)
df['dokumen_peraturan'] = df['dokumen_peraturan'].apply(ast.literal_eval)

In [4]:
df.tail(1)

,title,jenis_bentuk_peraturan,pemrakarsa,nomor,tahun,tentang,tempat_penetapan,ditetapkan_tanggal,status,dokumen_peraturan
59,Undang-undang Nomor 6 Tahun 2021 Tentang Angga...,UNDANG-UNDANG,PEMERINTAH PUSAT,6,2021,ANGGARAN PENDAPATAN DAN BELANJA NEGARA TAHUN A...,Jakarta,27 Oktober 2021,Berlaku,"[https://peraturan.go.id/files/uu6-2021bt.pdf,..."


In [5]:
import os
from requests.exceptions import RequestException

download_dir = UU_DATA_DIR
metadata_list = []

for index, row in df.iterrows():
  slugified_date = slugify(row['ditetapkan_tanggal'])

  for doc_url in row['dokumen_peraturan']:
      original_filename = os.path.basename(doc_url)
      filename, extension = os.path.splitext(original_filename)

      unique_id = generate_id()
      slugified_filename = slugify(filename)
      new_filename = f"uu_{slugified_date}_{slugified_filename}_{unique_id}{extension}"
      new_filepath = os.path.join(download_dir, new_filename)

      try:
          download_file(doc_url, new_filepath)
          print(f"Downloaded {doc_url} to {new_filepath}")
      except RequestException as e:
          print(f"Failed to download {doc_url}: {e}")
          continue
      
      metadata = create_metadata(row, doc_url, unique_id, new_filename)
      metadata_list.append(metadata)

      break

Downloaded https://peraturan.go.id/files/uu-no-2-tahun-2024.pdf to data_pool/uu\uu_25_april_2024_uu_no_2_tahun_2024_wwozywmo.pdf
Downloaded https://peraturan.go.id/files/uu-no-3-tahun-2024.pdf to data_pool/uu\uu_25_april_2024_uu_no_3_tahun_2024_ltazwtyn.pdf
Downloaded https://peraturan.go.id/files/uu-no-1-tahun-2024.pdf to data_pool/uu\uu_02_januari_2024_uu_no_1_tahun_2024_6qkbdjep.pdf
Downloaded https://peraturan.go.id/files/uu-no-21-tahun-2023.pdf to data_pool/uu\uu_31_oktober_2023_uu_no_21_tahun_2023_ebqonchb.pdf
Downloaded https://peraturan.go.id/files/uu-no-22-tahun-2023.pdf to data_pool/uu\uu_20_desember_2023_uu_no_22_tahun_2023_hinungao.pdf
Downloaded https://peraturan.go.id/files/uu-no-20-tahun-2023.pdf to data_pool/uu\uu_31_oktober_2023_uu_no_20_tahun_2023_kgnsjp9j.pdf
Downloaded https://peraturan.go.id/files/uu-no-17-tahun-2023.pdf to data_pool/uu\uu_08_agustus_2023_uu_no_17_tahun_2023_gusp6hkd.pdf
Downloaded https://peraturan.go.id/files/uu-no-19-tahun-2023.pdf to data_pool/

In [6]:
import json

with open(UU_METADATA, 'w') as json_file:
    json.dump(metadata_list, json_file, indent=4)

print("All files downloaded and metadata saved to JSON.")

All files downloaded and metadata saved to JSON.
